# Phase 11_a — Agentic Benchmark Matrix

`11_a` owns the long-running benchmark matrix for agentic spectral detection across multiple multi-hop QA datasets and models.

Outputs from this notebook are designed to be consumed by `11_b` for attribution and downstream analysis.


## Setup

In [ ]:
import os, sys, shutil

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes scipy scikit-learn')

from spectral_utils import (
    load_model, free_memory,
    extract_all_features, FEAT_NAMES,
    sw_var_peak_adaptive,
    boot_auc, best_nadler_on,
    load_agentic_multihop_dataset,
    run_react_episode, aggregate_trajectory,
    simulate_retrieve_tool, step_retrieved_supporting_fact,
    branching_entropy, categorize_failure_mode, categorize_failure_mode_v2,
)

import datasets  # noqa: F401
print('spectral_utils imported OK')
print(f'FEAT_NAMES ({len(FEAT_NAMES)}): {FEAT_NAMES}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted OK')


In [ ]:
import pickle
import numpy as np
import pandas as pd
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TEMP = 1.0
N_SAMPLES = 200
MAX_STEPS = 3
MAX_NEW_PER_STEP = 256
AGGS = ['min', 'avg', 'last']

MODELS = [
    ('qwen25_7b',      'Qwen/Qwen2.5-7B-Instruct',                  {'quantize_4bit': False}),
    ('deepseek_r1_7b', 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',   {'quantize_4bit': False}),
    # Phase 10 models added for better class balance and credible SOTA comparison:
    ('mistral24b',     'mistralai/Mistral-Small-24B-Instruct-2501',  {'quantize_4bit': False}),
    ('qwen72b',        'Qwen/Qwen2.5-72B-Instruct-AWQ',              {'quantize_4bit': False}),  # AWQ — run in fresh runtime with gptqmodel stub cell
]
DATASETS = ['hotpotqa', '2wikimultihopqa']

BASE_DIR = '/content/drive/MyDrive/hallucination_detection/cache/phase11_agentic_v2'
RAW_DIR = os.path.join(BASE_DIR, 'raw')
FEAT_DIR = os.path.join(BASE_DIR, 'features')
RES_DIR = os.path.join(BASE_DIR, 'results_a')
PLOT_DIR = os.path.join(BASE_DIR, 'plots_a')
MANIFEST_PATH = os.path.join(RES_DIR, 'phase11a_results.pkl')

for d in (RAW_DIR, FEAT_DIR, RES_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)

print('Models:', [m[0] for m in MODELS])
print('Datasets:', DATASETS)
print('Cache root:', BASE_DIR)

## Data

In [ ]:
ROWS_BY_DATASET = {
    dataset: load_agentic_multihop_dataset(dataset, n_samples=N_SAMPLES)
    for dataset in DATASETS
}
for dataset, rows in ROWS_BY_DATASET.items():
    row = rows[0]
    print(f'[{dataset}] rows={len(rows)}')
    print(f'  question: {row["question"][:140]}')
    print(f'  answer:   {row["answer"]}')
    print(f'  support titles: {row["supporting_facts"]["title"][:3]}')
    print(f'  context paragraphs: {len(row["context"]["title"])}')


In [ ]:
for dataset, rows in ROWS_BY_DATASET.items():
    row = rows[0]
    title, passage = simulate_retrieve_tool(row['question'], row['context'])
    print(f'[{dataset}] retrieved title: {title}')
    print(f'[{dataset}] is supporting: {step_retrieved_supporting_fact(title, row["supporting_facts"]["title"])}')
    print(f'[{dataset}] passage preview: {passage[:180]}')


## Inference

In [ ]:
from tqdm.auto import tqdm

def raw_path(model_key, dataset):
    return os.path.join(RAW_DIR, f'{model_key}__{dataset}.pkl')


def feat_path(model_key, dataset):
    return os.path.join(FEAT_DIR, f'{model_key}__{dataset}.pkl')


def run_agentic_inference(mdl, tok, model_key, dataset, rows, checkpoint_every=10):
    path = raw_path(model_key, dataset)
    results = []
    if os.path.exists(path):
        with open(path, 'rb') as f:
            results = pickle.load(f)
        print(f'[{model_key}/{dataset}] resumed from {len(results)} trajectories')
    start = len(results)
    if start >= len(rows):
        print(f'[{model_key}/{dataset}] already complete ({len(results)}/{len(rows)})')
        return
    for i in tqdm(range(start, len(rows)), desc=f'{model_key}/{dataset}'):
        row = rows[i]
        try:
            traj = run_react_episode(
                mdl, tok,
                question=row['question'],
                context=row['context'],
                supporting_titles=row['supporting_facts']['title'],
                gold_answer=row['answer'],
                T=TEMP,
                max_steps=MAX_STEPS,
                max_new_per_step=MAX_NEW_PER_STEP,
            )
            traj['idx'] = i
            traj['dataset'] = dataset
            traj['model_key'] = model_key
            results.append(traj)
        except Exception as ex:
            print(f'[{model_key}/{dataset}] error idx={i}: {ex}')
        if (i + 1) % checkpoint_every == 0:
            with open(path, 'wb') as f:
                pickle.dump(results, f)
    with open(path, 'wb') as f:
        pickle.dump(results, f)
    print(f'[{model_key}/{dataset}] saved {len(results)} trajectories to {path}')


def status_for_cell(model_key, dataset):
    path = raw_path(model_key, dataset)
    n = len(pickle.load(open(path, 'rb'))) if os.path.exists(path) else 0
    print(f'[{model_key}/{dataset}] status: {n}/{N_SAMPLES}')


In [ ]:
# ── Inference driver ──────────────────────────────────────────────────────────
# ONLY_MODEL_KEYS: set to a list to run a subset of models. None = run all.
#   Normal runtime (qwen25_7b + deepseek_r1_7b + mistral24b): set None or omit qwen72b
#   Fresh runtime for qwen72b only: set ONLY_MODEL_KEYS = ['qwen72b']
#     and run the gptqmodel stub cell above first.
# ONLY_DATASETS: set to ['hotpotqa'] or ['2wikimultihopqa'] to run one dataset.
ONLY_MODEL_KEYS = None
ONLY_DATASETS = None

selected_models = [m for m in MODELS if ONLY_MODEL_KEYS is None or m[0] in ONLY_MODEL_KEYS]
selected_datasets = [d for d in DATASETS if ONLY_DATASETS is None or d in ONLY_DATASETS]

for model_key, model_id, kw in selected_models:
    mdl, tok = load_model(model_id, **kw)
    print(f'Loaded {model_key}')
    for dataset in selected_datasets:
        run_agentic_inference(mdl, tok, model_key, dataset, ROWS_BY_DATASET[dataset])
        status_for_cell(model_key, dataset)
    del mdl, tok
    free_memory()

In [ ]:
ONLY_MODEL_KEYS = None
ONLY_DATASETS = None

selected_models = [m for m in MODELS if ONLY_MODEL_KEYS is None or m[0] in ONLY_MODEL_KEYS]
selected_datasets = [d for d in DATASETS if ONLY_DATASETS is None or d in ONLY_DATASETS]

for model_key, model_id, kw in selected_models:
    mdl, tok = load_model(model_id, **kw)
    print(f'Loaded {model_key}')
    for dataset in selected_datasets:
        run_agentic_inference(mdl, tok, model_key, dataset, ROWS_BY_DATASET[dataset])
        status_for_cell(model_key, dataset)
    del mdl, tok
    free_memory()


## Feature extraction

In [ ]:
FEAT_KEYS = None

def extract_for_cell(model_key, dataset, force=False):
    path = feat_path(model_key, dataset)
    if not force and os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)

    raw = pickle.load(open(raw_path(model_key, dataset), 'rb'))
    step_features = []
    step_verb_conf = []
    step_labels = []
    traj_labels = []
    failure_modes = []
    failure_modes_v2 = []

    for traj in raw:
        per_step_f = []
        per_step_c = []
        per_step_l = []
        for step in traj['steps']:
            ents = step.get('token_entropies') or []
            f = extract_all_features(ents) if len(ents) >= 8 else None
            if f is not None:
                f['sw_var_peak'] = sw_var_peak_adaptive(ents)  # adaptive window for 50-150 token steps
                f['branching_entropy'] = branching_entropy(ents, window=3)
            per_step_f.append(f)
            per_step_c.append(step.get('confidence'))
            per_step_l.append(bool(step.get('step_correct', False)))
        step_features.append(per_step_f)
        step_verb_conf.append(per_step_c)
        step_labels.append(per_step_l)
        traj_labels.append(bool(traj.get('trajectory_correct', False)))
        failure_modes.append(categorize_failure_mode(traj))
        failure_modes_v2.append(categorize_failure_mode_v2(traj))

    feat_keys = None
    for traj_f in step_features:
        for step_f in traj_f:
            if step_f is not None:
                feat_keys = list(step_f.keys())
                break
        if feat_keys:
            break
    if feat_keys is None:
        raise RuntimeError(f'No valid features extracted for {model_key}/{dataset}')

    traj_features = {agg: {} for agg in AGGS}
    n = len(traj_labels)
    for agg in AGGS:
        for key in feat_keys:
            arr = np.full(n, np.nan)
            for i, traj_f in enumerate(step_features):
                vals = [step_f[key] for step_f in traj_f if step_f is not None and key in step_f]
                if vals:
                    arr[i] = aggregate_trajectory(vals, agg=agg)
            traj_features[agg][key] = arr
        arr_c = np.full(n, np.nan)
        for i, vc in enumerate(step_verb_conf):
            vals = [v for v in vc if v is not None and not np.isnan(v)]
            if vals:
                arr_c[i] = aggregate_trajectory(vals, agg=agg)
        traj_features[agg]['verb_conf'] = arr_c

    cell = {
        'model_key': model_key,
        'dataset': dataset,
        'step_features': step_features,
        'step_verb_conf': step_verb_conf,
        'step_labels': step_labels,
        'traj_features': traj_features,
        'traj_labels': np.asarray(traj_labels, dtype=bool),
        'failure_modes': failure_modes,
        'failure_modes_v2': failure_modes_v2,
        'feature_keys': feat_keys,
        'n_traj': n,
    }
    with open(path, 'wb') as f:
        pickle.dump(cell, f)
    return cell


ALL_CELLS = {}
CELL_KEYS = []
for model_key, _, _ in MODELS:
    for dataset in DATASETS:
        if os.path.exists(raw_path(model_key, dataset)):
            key = (model_key, dataset)
            ALL_CELLS[key] = extract_for_cell(model_key, dataset)
            CELL_KEYS.append(key)

if not CELL_KEYS:
    raise RuntimeError('No completed raw trajectory files found. Run the inference cell first.')

CELL_KEYS = sorted(CELL_KEYS)
FEAT_KEYS = list(ALL_CELLS[CELL_KEYS[0]]['feature_keys'])
print('Cells:', CELL_KEYS)
print('Feature keys:', FEAT_KEYS)


## Core detector analysis

In [ ]:
g0_rows = []
for model_key, dataset in CELL_KEYS:
    cell = ALL_CELLS[(model_key, dataset)]
    labels = cell['traj_labels']
    n_pos = int(labels.sum())
    n_neg = int(len(labels) - n_pos)
    step_succ = [lab for traj_l in cell['step_labels'] for lab in traj_l]
    sr = (sum(step_succ) / len(step_succ)) if step_succ else 0.0
    g0_rows.append({
        'model': model_key,
        'dataset': dataset,
        'n_traj': len(labels),
        'n_correct': n_pos,
        'n_incorrect': n_neg,
        'mean_step_success': sr,
        'g0_n': len(labels) >= 150,
        'g0_classes': n_pos >= 10 and n_neg >= 10,
    })
df_g0 = pd.DataFrame(g0_rows)
print(df_g0.to_string(index=False))


In [ ]:
auc_records = []
for model_key, dataset in CELL_KEYS:
    cell = ALL_CELLS[(model_key, dataset)]
    labels = cell['traj_labels']
    for agg in AGGS:
        row = {'model': model_key, 'dataset': dataset, 'agg': agg}
        for key in FEAT_KEYS + ['verb_conf']:
            scores = cell['traj_features'][agg][key]
            mask = ~np.isnan(scores)
            if mask.sum() < 20:
                row[key] = float('nan')
                continue
            auc, _, _ = boot_auc(labels[mask], scores[mask])
            if not np.isnan(auc) and auc < 0.5:
                auc, _, _ = boot_auc(labels[mask], -scores[mask])
            row[key] = auc
        auc_records.append(row)
df_auc = pd.DataFrame(auc_records)
with pd.option_context('display.max_columns', None, 'display.width', 240):
    print(df_auc.to_string(index=False))


In [ ]:
NADLER_PATH = os.path.join(RES_DIR, 'phase11a_nadler_res.pkl')
FORCE_RECOMPUTE_NADLER = False

if not FORCE_RECOMPUTE_NADLER and os.path.exists(NADLER_PATH):
    with open(NADLER_PATH, 'rb') as f:
        NADLER_RES = pickle.load(f)
else:
    NADLER_RES = {}
    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        labels = cell['traj_labels']
        for agg in AGGS:
            keys = [key for key in FEAT_KEYS if key not in ('trace_length', 'pe_min')]
            feat_dict = {key: cell['traj_features'][agg][key] for key in keys}
            mask = np.all([~np.isnan(feat_dict[key]) for key in keys], axis=0)
            if mask.sum() < 30:
                NADLER_RES[(model_key, dataset, agg)] = {
                    'auc': float('nan'), 'lo': float('nan'), 'hi': float('nan'),
                    'subset': [], 'weights': [], 'sign_map': {}, 'n_used': int(mask.sum()),
                }
                continue
            feat_dict_m = {key: vals[mask] for key, vals in feat_dict.items()}
            labels_m = labels[mask]
            sign_map = {}
            for key in keys:
                ap, _, _ = boot_auc(labels_m, feat_dict_m[key])
                an, _, _ = boot_auc(labels_m, -feat_dict_m[key])
                sign_map[key] = +1 if (not np.isnan(ap) and ap >= an) else -1
            auc, lo, hi, subset, weights = best_nadler_on(
                feat_dict_m, keys, labels_m, max_size=4,
                label=f'{model_key}/{dataset}/{agg}', compare_mean=False,
            )
            NADLER_RES[(model_key, dataset, agg)] = {
                'auc': auc,
                'lo': lo,
                'hi': hi,
                'subset': list(subset) if subset else [],
                'weights': list(weights) if weights is not None else [],
                'sign_map': sign_map,
                'n_used': int(mask.sum()),
            }
    with open(NADLER_PATH, 'wb') as f:
        pickle.dump(NADLER_RES, f)

for key, res in sorted(NADLER_RES.items()):
    print(key, res['auc'], res['subset'])


In [ ]:
AUQ_PATH = os.path.join(RES_DIR, 'phase11a_auq_res.pkl')
FORCE_RECOMPUTE_AUQ = False

if not FORCE_RECOMPUTE_AUQ and os.path.exists(AUQ_PATH):
    with open(AUQ_PATH, 'rb') as f:
        AUQ_RES = pickle.load(f)
else:
    AUQ_RES = {}
    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        labels = cell['traj_labels']
        for agg in AGGS:
            scores = cell['traj_features'][agg]['verb_conf']
            mask = ~np.isnan(scores)
            if mask.sum() < 20:
                AUQ_RES[(model_key, dataset, agg)] = {'auc': float('nan'), 'lo': float('nan'), 'hi': float('nan'), 'n_used': int(mask.sum())}
                continue
            auc, lo, hi = boot_auc(labels[mask], scores[mask])
            if not np.isnan(auc) and auc < 0.5:
                auc, lo, hi = boot_auc(labels[mask], -scores[mask])
            AUQ_RES[(model_key, dataset, agg)] = {'auc': auc, 'lo': lo, 'hi': hi, 'n_used': int(mask.sum())}
    with open(AUQ_PATH, 'wb') as f:
        pickle.dump(AUQ_RES, f)

for key, res in sorted(AUQ_RES.items()):
    print(key, res['auc'])


In [ ]:
FUSION_PATH = os.path.join(RES_DIR, 'phase11a_fusion_res.pkl')
LEN_PATH = os.path.join(RES_DIR, 'phase11a_len_res.pkl')
PCA_PATH = os.path.join(RES_DIR, 'phase11a_pca_res.pkl')
RHO_PATH = os.path.join(RES_DIR, 'phase11a_rho_res.pkl')
FORCE_RECOMPUTE_EXTRA = False

if not FORCE_RECOMPUTE_EXTRA and all(os.path.exists(p) for p in (FUSION_PATH, LEN_PATH, PCA_PATH, RHO_PATH)):
    with open(FUSION_PATH, 'rb') as f:
        FUSION_RES = pickle.load(f)
    with open(LEN_PATH, 'rb') as f:
        LEN_RES = pickle.load(f)
    with open(PCA_PATH, 'rb') as f:
        PCA_RES = pickle.load(f)
    with open(RHO_PATH, 'rb') as f:
        RHO_RES = pickle.load(f)
else:
    from scipy.stats import spearmanr
    from sklearn.decomposition import PCA
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler

    FUSION_RES = {}
    LEN_RES = {}
    PCA_RES = {}
    RHO_RES = {}

    for model_key, dataset in CELL_KEYS:
        cell = ALL_CELLS[(model_key, dataset)]
        labels = cell['traj_labels']
        epr_steps = []
        conf_steps = []
        for traj_f, traj_c in zip(cell['step_features'], cell['step_verb_conf']):
            for step_f, conf in zip(traj_f, traj_c):
                if step_f is not None and 'epr' in step_f:
                    epr_steps.append(step_f['epr'])
                    conf_steps.append(conf)
        if len(epr_steps) >= 30:
            rho, p = spearmanr(epr_steps, conf_steps)
        else:
            rho, p = float('nan'), float('nan')
        RHO_RES[(model_key, dataset)] = {'rho': rho, 'p': p, 'n': len(epr_steps)}

        for agg in AGGS:
            fused_keys = [key for key in FEAT_KEYS if key != 'trace_length'] + ['verb_conf']
            feat_dict = {key: cell['traj_features'][agg][key] for key in fused_keys}
            mask = np.all([~np.isnan(feat_dict[key]) for key in fused_keys], axis=0)
            if mask.sum() < 30:
                FUSION_RES[(model_key, dataset, agg)] = {'auc': float('nan'), 'lo': float('nan'), 'hi': float('nan'), 'subset': [], 'weights': [], 'verb_conf_in_subset': False, 'n_used': int(mask.sum())}
            else:
                feat_dict_m = {key: vals[mask] for key, vals in feat_dict.items()}
                labels_m = labels[mask]
                auc, lo, hi, subset, weights = best_nadler_on(
                    feat_dict_m, fused_keys, labels_m, max_size=4,
                    label=f'{model_key}/{dataset}/{agg}-fused', compare_mean=False,
                )
                FUSION_RES[(model_key, dataset, agg)] = {
                    'auc': auc,
                    'lo': lo,
                    'hi': hi,
                    'subset': list(subset) if subset else [],
                    'weights': list(weights) if weights is not None else [],
                    'verb_conf_in_subset': bool(subset and 'verb_conf' in subset),
                    'n_used': int(mask.sum()),
                }

            tl = cell['traj_features'][agg]['trace_length']
            mask_tl = ~np.isnan(tl)
            if mask_tl.sum() < 20:
                LEN_RES[(model_key, dataset, agg)] = {}
            else:
                a_tl, lo_tl, hi_tl = boot_auc(labels[mask_tl], tl[mask_tl])
                if not np.isnan(a_tl) and a_tl < 0.5:
                    a_tl, lo_tl, hi_tl = boot_auc(labels[mask_tl], -tl[mask_tl])
                spectral_keys = [key for key in FEAT_KEYS if key != 'trace_length']
                feat_dict = {key: cell['traj_features'][agg][key] for key in spectral_keys}
                mask = np.all([~np.isnan(feat_dict[key]) for key in spectral_keys], axis=0)
                feat_dict_m = {key: vals[mask] for key, vals in feat_dict.items()}
                labels_m = labels[mask]
                a_sp, lo_sp, hi_sp, subset_sp, _ = best_nadler_on(
                    feat_dict_m, spectral_keys, labels_m, max_size=4,
                    label=f'{model_key}/{dataset}/{agg}-spectralonly', compare_mean=False,
                )
                LEN_RES[(model_key, dataset, agg)] = {
                    'trace_length_auc': a_tl,
                    'trace_length_ci': (lo_tl, hi_tl),
                    'spectral_only_auc': a_sp,
                    'spectral_only_ci': (lo_sp, hi_sp),
                    'spectral_only_subset': list(subset_sp) if subset_sp else [],
                    'lift_over_length_pp': 100 * (a_sp - a_tl),
                }

            pca_keys = [key for key in FEAT_KEYS if key != 'trace_length']
            X = np.column_stack([cell['traj_features'][agg][key] for key in pca_keys])
            mask = ~np.any(np.isnan(X), axis=1)
            if mask.sum() < 30:
                PCA_RES[(model_key, dataset, agg)] = {}
            else:
                Xm = X[mask]
                ym = labels[mask]
                Xs = StandardScaler().fit_transform(Xm)
                pca = PCA(n_components=min(5, Xm.shape[1])).fit(Xs)
                pc1 = pca.transform(Xs)[:, 0]
                pc1_auc = roc_auc_score(ym, pc1)
                if pc1_auc < 0.5:
                    pc1_auc = roc_auc_score(ym, -pc1)
                PCA_RES[(model_key, dataset, agg)] = {
                    'pc1_auc': pc1_auc,
                    'pc1_var_ratio': float(pca.explained_variance_ratio_[0]),
                }

    with open(FUSION_PATH, 'wb') as f:
        pickle.dump(FUSION_RES, f)
    with open(LEN_PATH, 'wb') as f:
        pickle.dump(LEN_RES, f)
    with open(PCA_PATH, 'wb') as f:
        pickle.dump(PCA_RES, f)
    with open(RHO_PATH, 'wb') as f:
        pickle.dump(RHO_RES, f)

print('FUSION cells:', len(FUSION_RES))
print('LEN cells:', len(LEN_RES))
print('PCA cells:', len(PCA_RES))
print('RHO cells:', len(RHO_RES))


## Headline summary

In [ ]:
headline_rows = []
for model_key, dataset in CELL_KEYS:
    cell = ALL_CELLS[(model_key, dataset)]
    labels = cell['traj_labels']
    for agg in AGGS:
        epr_scores = cell['traj_features'][agg]['epr']
        mask = ~np.isnan(epr_scores)
        if mask.sum() >= 20:
            epr_auc, _, _ = boot_auc(labels[mask], epr_scores[mask])
            if not np.isnan(epr_auc) and epr_auc < 0.5:
                epr_auc, _, _ = boot_auc(labels[mask], -epr_scores[mask])
        else:
            epr_auc = float('nan')
        headline_rows.append({
            'model': model_key,
            'dataset': dataset,
            'agg': f'Φ_{agg}',
            'EPR-only': epr_auc,
            'AUQ': AUQ_RES.get((model_key, dataset, agg), {}).get('auc', float('nan')),
            'Spectral Nadler': NADLER_RES.get((model_key, dataset, agg), {}).get('auc', float('nan')),
            'Spectral+AUQ': FUSION_RES.get((model_key, dataset, agg), {}).get('auc', float('nan')),
        })

df_headline = pd.DataFrame(headline_rows)
with pd.option_context('display.float_format', lambda x: f'{100*x:.1f}%' if isinstance(x, float) and not np.isnan(x) else '  -  '):
    print(df_headline.to_string(index=False))

g0_pass = bool((df_g0['g0_n'] & df_g0['g0_classes']).all())
g1_pass = any(abs(res['rho']) < 0.5 for res in RHO_RES.values() if not np.isnan(res['rho']))
g2_vals = [res['auc'] for key, res in NADLER_RES.items() if key[2] == 'min' and not np.isnan(res['auc'])]
g2_best = max(g2_vals) if g2_vals else float('nan')
g2_pass = any(v >= 0.70 for v in g2_vals)
g3_diffs = []
for key, res in FUSION_RES.items():
    fused_auc = res.get('auc', float('nan'))
    auq_auc = AUQ_RES.get(key, {}).get('auc', float('nan'))
    if not (np.isnan(fused_auc) or np.isnan(auq_auc)):
        g3_diffs.append(fused_auc - auq_auc)
g3_best = max(g3_diffs) if g3_diffs else float('nan')
g3_pass = any(v >= 0.03 for v in g3_diffs)
g4_diffs = [res['spectral_only_auc'] - res['trace_length_auc'] for res in LEN_RES.values() if res]
g4_best = max(g4_diffs) if g4_diffs else float('nan')
g4_pass = any(v >= 0.03 for v in g4_diffs)
g5_vals = [res.get('auc', float('nan')) for key, res in FUSION_RES.items() if key[2] == 'min']
g5_best = max([v for v in g5_vals if not np.isnan(v)], default=float('nan'))
g5_pass = any(v >= 0.791 for v in g5_vals if not np.isnan(v))

print('G0', g0_pass)
print('G1', g1_pass)
print('G2', g2_pass, g2_best)
print('G3', g3_pass, g3_best)
print('G4', g4_pass, g4_best)
print('G5', g5_pass, g5_best)


In [ ]:
results = {
    'config': {
        'seed': SEED,
        'models': MODELS,
        'datasets': DATASETS,
        'aggs': AGGS,
        'n_samples': N_SAMPLES,
        'max_steps': MAX_STEPS,
        'temperature': TEMP,
        'base_dir': BASE_DIR,
        'raw_dir': RAW_DIR,
        'feat_dir': FEAT_DIR,
        'res_dir': RES_DIR,
        'plot_dir': PLOT_DIR,
    },
    'cell_keys': CELL_KEYS,
    'feature_keys': FEAT_KEYS,
    'NADLER_RES': NADLER_RES,
    'AUQ_RES': AUQ_RES,
    'FUSION_RES': FUSION_RES,
    'LEN_RES': LEN_RES,
    'PCA_RES': PCA_RES,
    'RHO_RES': RHO_RES,
    'headline_table': df_headline.to_dict(orient='records'),
    'gates': {
        'g0': g0_pass,
        'g1': g1_pass,
        'g2': g2_pass,
        'g3': g3_pass,
        'g4': g4_pass,
        'g5': g5_pass,
    },
}
with open(MANIFEST_PATH, 'wb') as f:
    pickle.dump(results, f)
print('Saved', MANIFEST_PATH)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
heatmap_cols = ['EPR-only', 'AUQ', 'Spectral Nadler', 'Spectral+AUQ']
heatmap = df_headline[heatmap_cols].values
ax.imshow(heatmap, aspect='auto', cmap='viridis', vmin=0.5, vmax=0.9)
ax.set_xticks(range(len(heatmap_cols)))
ax.set_xticklabels(heatmap_cols, rotation=20, ha='right')
ax.set_yticks(range(len(df_headline)))
ax.set_yticklabels([f"{row['model']}/{row['dataset']}/{row['agg']}" for _, row in df_headline.iterrows()])
ax.set_title('Phase 11_a detector AUC heatmap')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'phase11a_detector_heatmap.png'), dpi=120)
plt.show()

rho_items = list(RHO_RES.items())
fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [f'{mk}/{ds}' for (mk, ds), _ in rho_items]
vals = [res['rho'] for _, res in rho_items]
ax.bar(labels, vals, color=['steelblue' if abs(v) < 0.5 else 'crimson' for v in vals])
ax.axhline(0.5, color='red', ls='--', lw=1)
ax.axhline(-0.5, color='red', ls='--', lw=1)
ax.axhline(0.0, color='black', lw=0.5)
ax.set_ylabel('Spearman ρ')
ax.set_title('ρ(EPR_step, verb_conf_step) per (model, dataset)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'phase11a_rho.png'), dpi=120)
plt.show()
